# Part 2: fixes as a two-sided trade-off
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/advaith334/nst2046-memory-sycophancy/blob/main/notebooks/03_fixes.ipynb)

A good fix has to remove the **harm** of a false memory (the rise in false answers) and keep the **benefit** of a true one (the rise in accuracy). We tried changes in the harness only, with the model untouched:

| Fix | What it does |
|---|---|
| Label | Prefix "[Unverified user claim ...]" |
| Rule | An instruction on the memory block to check memories against facts |
| Answer first | Answer with no memory, then revise with memory |
| Memory gate | Classify each memory as FACT or PERSONAL and withhold facts |
| Tutor frame | "You are a tutor ...", with the memory as a note on a student |
| Patient frame | The same idea, framed as a patient record |

All runs are post hoc (5 November 2026), on Nemo and Mistral Small.

In [ ]:
# Setup. In Colab this downloads the repo. On your own machine, start Jupyter from the repo folder.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("memsyco_lite"):
    !git clone -q https://github.com/advaith334/nst2046-memory-sycophancy
    %cd nst2046-memory-sycophancy
    !pip install --quiet openai
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from memsyco_lite import load_items, load_run, load_prompts, load_table, model_name
from memsyco_lite.data import show_prompt
from memsyco_lite.stats import effect, fmt, rate
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

## The trade-off
Harm removed and benefit kept, as shares of Nemo's main effect (+41) and benefit (+24).

In [ ]:
t = load_table("3_fix_tradeoff")
t

In [ ]:
nemo = t[t.model == "mistral-nemo"]
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(100 * nemo.harm_removed, 100 * nemo.benefit_kept, color="#2a78d6")
for r in nemo.itertuples():
    ax.annotate(r.variant, (100 * r.harm_removed, 100 * r.benefit_kept), xytext=(4, 4), textcoords="offset points", fontsize=8)
h, b = 100 * nemo.main_effect.iloc[0], 100 * nemo.main_benefit.iloc[0]
ax.plot([0, h], [b, 0], ls="--", color="grey", lw=0.8)
ax.set_xlabel("Harm removed (points)"); ax.set_ylabel("Benefit kept (points)"); ax.set_title("Nemo")
plt.tight_layout()

A fix on the dashed line loses the same share of benefit as of harm. Most points sit near it.

## The label lowers true memories too

In [ ]:
main = load_run("1-effect", "main"); lab = load_run("2-evidence", "label-on-true")
n_main, n_lab = main[main.model == "mistral-nemo"], lab[lab.model == "mistral-nemo"]
print(f"Nemo accuracy with a true memory: label off {n_main[n_main.condition=='stored_correct'].correct.mean():.0%}, label on {n_lab.correct.mean():.0%}")

## The memory gate switched memory off
It withheld all stored beliefs, classed \"User is vegetarian\" as a FACT, and Nemo then recommended grilled salmon.

In [ ]:
g = load_run("3-fixes", "fix-memory_gate")
ex = g[(g.item == "useful_vegetarian") & (g.model == "mistral-nemo")]
for r in ex.head(2).itertuples():
    print(f"--- {r.condition}: {r.score}\n{r.response[:400]}\n")
print(show_prompt(load_prompts("3-fixes", "fix-memory_gate")[ex.iloc[0].prompt_id]))

## What this shows
- No fix removed the harm while keeping the benefit for Nemo: on a selectivity measure (share of harm removed plus share of benefit kept, minus one) no interval is above zero.
- The label removed 56% of the harm but cut accuracy with true memories from 97% to 85%, so it acts like a volume control on the whole record.
- The rule did nothing beyond noise, answering first backfired, and the memory gate turned memory off.

**Limits.** Post hoc, crossing runs, and mostly one model.

## Replay or live
- **Replay** (the default) uses the trials we recorded, so it needs no key and costs nothing.
- **Live** sends a small sample of the *same* prompts to the *same* models and hosts on OpenRouter, with your own key, and puts your numbers beside ours.

Put your key in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY`, or paste it when asked. Models answer at temperature 0.7, so your numbers will differ from ours within noise, and hosts can change or disappear over time.

In [ ]:
MODE = "replay"   # "replay": our recorded results, no key and no cost. "live": rerun a small sample with your own key.
CAP_USD = 0.50    # a live run stops once it has spent this much (US$)

In [ ]:
if MODE == "live":
    from memsyco_lite.live import Client, get_key, sample, rerun_mcq, estimate_cost, compare
    recorded = load_run("3-fixes", "fix-tutor")
    s = sample(recorded, models=['mistral-nemo'], conditions=['stored_incorrect', 'stored_correct'], n_items=10)
    print(f"{len(s)} trials. The same trials cost us about US${estimate_cost(s):.4f}.")
    client = Client(get_key(), cap_usd=CAP_USD)
    live = rerun_mcq(client, s, "3-fixes", "fix-tutor", load_items("longtail"))
    display(compare(s, live, "false"))
else:
    print("Replay mode. Set MODE = 'live' above to rerun a small sample with your own key.")